# <b>SN-AI Project</b>

## <b>Project Objective</b>

Given a dataset containing all the **Serie A match schedules and results** from the **2017/2018 season to the 2024/2025 season**, the goal of this project is to build a **neural network model capable of predicting the results of Serie A matches for the 2025/2026 season**.

The model will be trained using historical match data in **JSON format**, which contains information such as:

- Home team  
- Away team  
- Goals scored by the home team  
- Goals scored by the away team  
- Match date  
- Stadium where the match was played  

### Example of a match entry in the dataset

```json
{
  "MatchNumber": 1,
  "RoundNumber": 1,
  "DateUtc": "2017-08-19 16:00:00Z",
  "Location": "Juventus Stadium",
  "HomeTeam": "Juventus",
  "AwayTeam": "Cagliari",
  "Group": null,
  "HomeTeamScore": 3,
  "AwayTeamScore": 0
}
```

### Expected Output
The expected outputs are the results of the matches in the Serie A 2025/26 schedule. Our goal is to compare them with the actual match results and see how many times it predicts correctly.

**Premise:** we know that real results are not based only on past results; there are many other factors that can influence them, such as the physical condition of the players on the field, yearly squad changes, and other factors.



In [9]:
from pathlib import Path

def read_file(file_path: Path) -> str:
    """Read and return the content of a file."""
    print(f"Reading file: {file_path}")
    return file_path.read_text(encoding="utf-8")


def read_folder_files(folder_path: str) -> dict:
    """Read all files in a folder and return a dict {filename: content}."""
    folder = Path(folder_path)

    return {
        file.name: read_file(file)
        for file in folder.iterdir()
        if file.is_file()
    }


train_path = "/Users/brunovincenzi/Coding/Università/SN-AI/json_files/train"
eval_path = "/Users/brunovincenzi/Coding/Università/SN-AI/json_files/eval"

train_file_contents = read_folder_files(train_path)
eval_file_contents = read_folder_files(eval_path)


Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_1516.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_1920.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_1819.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_2021.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_1617.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_2122.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/train/SerieA_1718.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/eval/SerieA_2526.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/eval/SerieA_2223.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/eval/SerieA_2425.json
Reading file: /Users/brunovincenzi/Coding/Università/SN-AI/json_files/eva

In [10]:
import json
from collections import defaultdict


def clean_match(match: dict) -> dict:
    """Remove unnecessary keys from match."""
    keys_to_remove = {"Location", "Group", "DateUtc"}
    return {k: v for k, v in match.items() if k not in keys_to_remove}


def update_team_points(match: dict, teams: dict):
    """Update team points based on match result."""
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    teams.setdefault(home, 0)
    teams.setdefault(away, 0)

    if home_score > away_score:
        teams[home] += 3
    elif home_score < away_score:
        teams[away] += 3
    else:
        teams[home] += 1
        teams[away] += 1


def track_past_match(match: dict, past_matches: dict):
    """Store past match results."""
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    key = f"{home}-{away}"
    result = f"{home_score}-{away_score}"

    past_matches[key].append(result)


def parse_json(json_content: str, teams: dict, past_matches: dict):
    """Parse JSON content and update statistics."""
    try:
        matches = json.loads(json_content)

        if not isinstance(matches, list):
            print("Expected a list of matches")
            return []

        cleaned_matches = []

        for match in matches:
            match = clean_match(match)

            if match["HomeTeamScore"] is not None and match["AwayTeamScore"] is not None:
                update_team_points(match, teams)
                track_past_match(match, past_matches)

            cleaned_matches.append(match)

        return cleaned_matches

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []


def parse_dataset(file_contents: dict, teams: dict, past_matches: dict):
    """Parse all files in a dataset."""
    parsed_contents = {}

    for file, content in file_contents.items():
        parsed_contents[file] = parse_json(content, teams, past_matches)

    return parsed_contents


# Structures
all_possible_past_matches = defaultdict(list)
all_possible_teams = {}

# Parse datasets
train_parsed_contents = parse_dataset(
    train_file_contents, all_possible_teams, all_possible_past_matches
)

eval_parsed_contents = parse_dataset(
    eval_file_contents, all_possible_teams, all_possible_past_matches
)

# Print results
print("All possible past matches:")
for match_key, results in all_possible_past_matches.items():
    print(f"{match_key}: {results}")

print("\nAll possible teams and their points:")
for team, points in all_possible_teams.items():
    print(f"{team}: {points} points")

All possible past matches:
Hellas Verona-Roma: ['1-1', '0-0', '0-0', '3-2', '0-1', '1-3', '3-2', '3-2']
Lazio-Bologna: ['2-1', '2-1', '3-3', '2-1', '1-1', '3-0', '1-1', '1-1', '2-1', '3-0', '3-0']
Juventus-Udinese: ['0-1', '4-1', '4-1', '4-1', '2-1', '2-0', '2-0', '3-1', '1-0', '2-0', '2-0']
Empoli-Chievoverona: ['1-3', '2-2', '0-0']
Fiorentina-Milan: ['2-0', '2-3', '0-1', '2-3', '0-0', '4-3', '1-1', '1-1', '2-1', '2-1', '2-1']
Frosinone-Torino: ['1-2', '1-2']
Inter-Atalanta: ['1-0', '1-0', '0-0', '1-0', '7-1', '2-2', '2-0', '3-2', '4-0', '4-0']
Palermo-Genoa: ['1-0', '1-0']
Sampdoria-Carpi: ['5-2']
Sassuolo-Napoli: ['2-1', '3-3', '1-1', '3-3', '2-2', '2-2', '1-1', '0-2', '0-2']
Bologna-Sassuolo: ['0-1', '3-4', '2-1', '3-4', '1-1', '1-3', '2-1', '1-1', '3-0']
Milan-Empoli: ['2-1', '3-0', '1-2', '1-0', '0-0', '3-0', '3-0']
Roma-Juventus: ['2-1', '2-2', '2-0', '2-2', '3-1', '3-4', '0-0', '3-3', '1-0', '1-1', '1-1']
Atalanta-Frosinone: ['2-0', '4-0']
Carpi-Inter: ['1-2']
Chievoverona-Lazi